# Multi-Agent University Student Pipeline

**Author:** asket  
**Location:** `week8/community_contributions/asket/`

This notebook implements a **three-stage AI pipeline for university students only** (course questions, assignments, exam prep). Inspired by **[Klingbo.com](https://klingbo.com)** — Ghana's AI-powered education platform.

**For university students only:** assignment help, exam revision, course content — no legal or non-academic use.  

**Pipeline stages:**  
1. **Query Expander** — turns one course question into 4–5 search keyphrases for retrieval.  
2. **Academic Researcher** — retrieves relevant chunks from course materials (Chroma RAG).  
3. **Student Advisor** — synthesizes a clear answer from the retrieved context (litellm/OpenAI).  

**Run from `week8/`** so that the asket pipeline resolves correctly.

## Setup — run from week8/

Set working directory to **week8** and add **asket** to the path so we can import `student_pipeline`.

In [42]:
import os
import sys
from pathlib import Path

cwd = Path.cwd()
if (cwd / "agents").exists() and (cwd / "deal_agent_framework.py").exists():
    week8_dir = cwd
elif (cwd.parent.parent / "agents").exists():
    week8_dir = cwd.parent.parent
else:
    week8_dir = cwd
if str(week8_dir) not in sys.path:
    sys.path.insert(0, str(week8_dir))
sys.path = [str(week8_dir)] + [p for p in sys.path if p != str(week8_dir) and "community_contributions/asket" not in p]
asket_dir = week8_dir / "community_contributions" / "asket"
if asket_dir.exists() and str(asket_dir) not in sys.path:
    sys.path.insert(0, str(asket_dir))
os.chdir(week8_dir)
print("Working directory:", os.getcwd())

from dotenv import load_dotenv
load_dotenv(override=True)
import logging
logging.getLogger().setLevel(logging.INFO)

Working directory: /Users/franckasket/Documents/GitHub/llm_engineering2/week8


### Install dependencies (run once)

Run this cell once if you see missing-module errors; then **Kernel → Restart** and re-run from Setup.

In [43]:
%pip install -q litellm chromadb gradio modal openai python-dotenv sentence-transformers --user --break-system-packages

15980.85s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Note: you may need to restart the kernel to use updated packages.


## Build the RAG index

Index the **knowledge_base** (e.g. `knowledge_base/education_sample.txt`) into Chroma. The sample includes calculus, contract law, study skills, and WASSCE/university prep — Klingbo-style content.

In [44]:
import student_pipeline

client = student_pipeline.build_index(force=False)
coll = client.get_collection(student_pipeline.COLLECTION_NAME)
print("Chunks in index:", coll.count())

Chunks in index: 4


## Run the pipeline once

Ask a question. You'll see **thinking** (expander → researcher → tutor) and the **final answer**. Set `use_modal_expander=True` if you've deployed `modal_student_expander.py` to Modal.

In [46]:
import importlib
import student_pipeline
importlib.reload(student_pipeline)  # ensure latest run_pipeline (returns 4 values)

question = "What is consideration in contract law?"
use_modal = False  # Set True after: modal deploy -m community_contributions/asket/modal_student_expander.py

expander_out, researcher_out, answer, status = student_pipeline.run_pipeline(question, use_modal_expander=use_modal, client=client)
print("Status:", status)
print("\n=== Agent 1: Query Expander (search keyphrases) ===")
print(expander_out)
print("\n=== Agent 2: Academic Researcher (retrieved course materials) ===")
print(researcher_out[:800] + "..." if len(researcher_out) > 800 else researcher_out)
print("\n=== Agent 3: Student Advisor (final answer) ===")
print(answer)

INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: mps


[2026-03-06 16:50:46 +0000] [Agents] [INFO] Use pytorch device_name: mps
[2026-03-06 16:50:46 +0000] [Agents] [INFO] Use pytorch device_name: mps
[2026-03-06 16:50:46 +0000] [Agents] [INFO] Use pytorch device_name: mps
[2026-03-06 16:50:46 +0000] [Agents] [INFO] Use pytorch device_name: mps


INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


[2026-03-06 16:50:46 +0000] [Agents] [INFO] Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
[2026-03-06 16:50:46 +0000] [Agents] [INFO] Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
[2026-03-06 16:50:46 +0000] [Agents] [INFO] Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
[2026-03-06 16:50:46 +0000] [Agents] [INFO] Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3547.32it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 1/1 [00:00<00:00, 124.69it/s]
16:50:54 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM completion() model= gpt-4o-mini; provider = openai
INFO:LiteLLM:
LiteLLM completion() model= gpt-4o-mini; provider = openai


[2026-03-06 16:50:54 +0000] [Agents] [INFO] 
LiteLLM completion() model= gpt-4o-mini; provider = openai
[2026-03-06 16:50:54 +0000] [Agents] [INFO] 
LiteLLM completion() model= gpt-4o-mini; provider = openai
[2026-03-06 16:50:54 +0000] [Agents] [INFO] 
LiteLLM completion() model= gpt-4o-mini; provider = openai
[2026-03-06 16:50:54 +0000] [Agents] [INFO] 
LiteLLM completion() model= gpt-4o-mini; provider = openai


16:50:56 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler


[2026-03-06 16:50:56 +0000] [Agents] [INFO] Wrapper: Completed Call, calling success_handler
[2026-03-06 16:50:56 +0000] [Agents] [INFO] Wrapper: Completed Call, calling success_handler
[2026-03-06 16:50:56 +0000] [Agents] [INFO] Wrapper: Completed Call, calling success_handler
[2026-03-06 16:50:56 +0000] [Agents] [INFO] Wrapper: Completed Call, calling success_handler
Status: Complete

=== Agent 1: Query Expander (search keyphrases) ===
  • "What is consideration in contract law"
  • "What is consideration in contract law course"
  • "What is consideration in contract law definition"
  • "What is consideration in contract law exam"
  • "What is consideration in contract law notes"

=== Agent 2: Academic Researcher (retrieved course materials) ===
Contract Law: Consideration
Consideration is something of value exchanged between parties to a contract. It can be money, a promise, or an act. In common law (including Ghana), a contract generally requires offer, acceptance, and consideratio

## Gradio UI — Multi-Agent University Student Pipeline

Three-panel interface: **Query Expander** → **Academic Researcher** → **Student Advisor**. For university course questions only. Requires **OPENAI_API_KEY** (or litellm config) for the Student Advisor step.

In [ ]:
import gradio as gr
import student_pipeline

_client = student_pipeline.build_index()

def run_agents(question: str, use_modal: bool, progress=gr.Progress()):
    if not question or not question.strip():
        return "", "", "", "Enter a course question."
    progress(0.2, desc="Running pipeline…")
    expander_out, researcher_out, answer, status = student_pipeline.run_pipeline(question, use_modal_expander=use_modal, client=_client)
    return expander_out, researcher_out, answer, f"System Status: {status}"

with gr.Blocks(title="Multi-Agent University Student Pipeline", theme=gr.themes.Soft()) as ui:
    gr.Markdown("## Multi-Agent University Student Pipeline")
    gr.Markdown("**For university students only.** Course questions, assignments, exam prep. Inspired by [Klingbo](https://klingbo.com).")
    status_out = gr.Textbox(label="System Status", value="Ready", interactive=False)
    inp = gr.Textbox(label="Enter your course question:", placeholder="e.g. What is consideration in contract law?")
    use_modal = gr.Checkbox(label="Use Modal query expander", value=False)
    run_btn = gr.Button("Run Agents")
    gr.Markdown("---")
    with gr.Row():
        expander_out = gr.Textbox(label="Agent 1: Query Expander — Search keyphrases", lines=6, interactive=False)
    with gr.Row():
        researcher_out = gr.Textbox(label="Agent 2: Academic Researcher — Retrieved course materials", lines=8, interactive=False)
    with gr.Row():
        answer_out = gr.Textbox(label="Agent 3: Student Advisor — Final answer", lines=8, interactive=False)
    run_btn.click(fn=run_agents, inputs=[inp, use_modal], outputs=[expander_out, researcher_out, answer_out, status_out])

ui.launch()

/var/folders/lp/dhfxmtbn53qg3w3jd5_ym5cc0000gn/T/ipykernel_56179/4149169960.py:13: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Multi-Agent University Student Pipeline", theme=gr.themes.Soft()) as ui:


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


INFO:sentence_transformers.SentenceTransformer:Use pytorch device_name: mps


[2026-03-06 16:52:47 +0000] [Agents] [INFO] Use pytorch device_name: mps
[2026-03-06 16:52:47 +0000] [Agents] [INFO] Use pytorch device_name: mps
[2026-03-06 16:52:47 +0000] [Agents] [INFO] Use pytorch device_name: mps
[2026-03-06 16:52:47 +0000] [Agents] [INFO] Use pytorch device_name: mps


INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


...

[2026-03-06 16:54:11 +0000] [Agents] [INFO] Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
[2026-03-06 16:54:11 +0000] [Agents] [INFO] Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
[2026-03-06 16:54:11 +0000] [Agents] [INFO] Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2
[2026-03-06 16:54:11 +0000] [Agents] [INFO] Load pretrained SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5268.46it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 1/1 [00:00<00:00, 112.94it/s]
16:54:18 - LiteLLM:INFO: utils.py:3898 - 
LiteLLM completion() model= gpt-4o-mini; provider = openai
INFO:LiteLLM:
LiteLLM completion() model= gpt-4o-mini; provider = openai


[2026-03-06 16:54:18 +0000] [Agents] [INFO] 
LiteLLM completion() model= gpt-4o-mini; provider = openai
[2026-03-06 16:54:18 +0000] [Agents] [INFO] 
LiteLLM completion() model= gpt-4o-mini; provider = openai
[2026-03-06 16:54:18 +0000] [Agents] [INFO] 
LiteLLM completion() model= gpt-4o-mini; provider = openai
[2026-03-06 16:54:18 +0000] [Agents] [INFO] 
LiteLLM completion() model= gpt-4o-mini; provider = openai


16:54:24 - LiteLLM:INFO: utils.py:1630 - Wrapper: Completed Call, calling success_handler
INFO:LiteLLM:Wrapper: Completed Call, calling success_handler


[2026-03-06 16:54:24 +0000] [Agents] [INFO] Wrapper: Completed Call, calling success_handler
[2026-03-06 16:54:24 +0000] [Agents] [INFO] Wrapper: Completed Call, calling success_handler
[2026-03-06 16:54:24 +0000] [Agents] [INFO] Wrapper: Completed Call, calling success_handler
[2026-03-06 16:54:24 +0000] [Agents] [INFO] Wrapper: Completed Call, calling success_handler


---

## Summary

- **Query Expander:** one course question → 4–5 search keyphrases (local or Modal).  
- **Academic Researcher:** Chroma RAG over `knowledge_base/` (course materials only).  
- **Student Advisor:** litellm/OpenAI generates the final answer for university students.  
- **Gradio UI:** three-panel layout (Expander → Researcher → Student Advisor). **University students only.**  

**Optional:** Deploy the expander to Modal:  
`modal deploy -m community_contributions/asket/modal_student_expander.py` from **week8/**.